In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint

pairs = [
    ("AAPL", "MSFT"),
    ("NVDA", "AMD"),
    ("KO", "PEP"),
    ("XOM", "CVX"),
    ("JPM", "BAC")
]

price_data = pd.read_csv("../data/prices.csv", index_col = 0, parse_dates =True).dropna()
price_data.head()

# Split data 
split_idx = int(len(price_data) * 0.5) 
split_date = price_data.index[split_idx]

train = price_data.loc[:split_date]
test = price_data.loc[split_date:]

In [5]:
# Cointegration test function
def test_cointegration(price_data, pairs):

    coint_results = []

    for (t1,t2) in pairs:

        x = np.log(price_data[t1])
        y = np.log(price_data[t2])

        score, pvalue, _ = coint(y,x)
        coint_results.append((t1,t2,pvalue))

    results_df = pd.DataFrame(coint_results, columns = ["Asset1", "Asset2", "p_value"])

    return results_df.sort_values("p_value")

In [6]:
# Run our cointegration test function
coint_df = test_cointegration(price_data, pairs)
print(coint_df)

  Asset1 Asset2   p_value
3    XOM    CVX  0.037721
4    JPM    BAC  0.082354
2     KO    PEP  0.314321
1   NVDA    AMD  0.556731
0   AAPL   MSFT  0.807370


In [7]:
# Filter by significant pairs
significant_pairs = [
    (row.Asset1, row.Asset2) for _, row in coint_df.iterrows() if row.p_value < 0.05
]

print("Significant Pairs:", significant_pairs)

Significant Pairs: [('XOM', 'CVX')]


### What we are doing
We are computing a line of best fit by using OLS, the testing the residuals for the presence of a unit root via ADF. If our residuals are stationary, we have found a cointegrated pair. In this case, XOM and CVX are cointegrated. (Note in previous days, we have found through backtesting that these pairs are not economically profitable hence cointegration does not equal profit)


## Day 23
### Note
Initially we had used a static hedge ratio, however relationships can drift, so we must manage risk dynamically instead of a static, one-time hedge.

In [8]:
# Rolling Hedge Ration function
def rolling_hedge_ratio(price_data, t1, t2, window):
    # log prices
    x = np.log(price_data[t1])
    y = np.log(price_data[t2])

    # rolling hedge ratio
    betas = []
    for i in range(len(price_data)):
        if i < window:
            betas.append(np.nan)
        else:
            x_window = x.iloc[i-window:i]
            y_window = y.iloc[i-window:i]

            X = add_constant(x_window)
            model = OLS(y_window,X).fit()
            betas.append(model.params[1])
        return pd.Series(betas, index = price_data.index)

In [9]:
# Backtest with rolling beta
def pair_returns_rolling_beta(price_data, t1, t2, beta_window, z_window, entry_z, cost=0.0005):

    # log prices
    x = np.log(price_data[t1])
    y = np.log(price_data[t2])

    beta_series = rolling_hedge_ratio(price_data, t1, t2, beta_window)

    spread = y - beta_series * x

    mean = spread.rolling(z_window).mean()
    std = spread.rolling(z_window).std()
    zscore = (spread - mean) / std

    position = pd.Series(0, index=spread.index)
    position[zscore > entry_z] = -1
    position[zscore < -entry_z] = 1
    position = position.ffill().fillna(0)

    spread_ret = spread.diff()
    strategy_ret = position.shift(1) * spread_ret

    trades = position.diff().abs()
    strategy_ret = strategy_ret - trades * cost

    return strategy_ret.fillna(0)    

In [22]:
entry_z = 1.0  # threshold for entering trades
beta_window = 60
z_window = 20
cost = 0.0005

rolling_returns = pd.DataFrame(index=price_data.index)

for (t1, t2) in significant_pairs:
    r = pair_returns_rolling_beta(
        price_data, 
        t1, t2,
        beta_window=beta_window,
        z_window=z_window,
        entry_z=entry_z,
        cost=cost
    )
    rolling_returns[f"{t1}_{t2}"] = r

rolling_returns["Portfolio"] = rolling_returns.mean(axis=1)

ValueError: Length of values (1) does not match length of index (785)

In [23]:
def performance_stats(returns):
    sharpe = returns.mean() / returns.std() * np.sqrt(252)
    cagr = (1 + returns.mean())**252 - 1
    max_dd = (returns.cumsum() - returns.cumsum().cummax()).min()
    return sharpe, cagr, max_dd

sharpe_rb, cagr_rb, dd_rb = performance_stats(rolling_returns["Portfolio"])

print("Rolling Beta Sharpe:", sharpe_rb)
print("Rolling Beta CAGR:", cagr_rb)
print("Rolling Beta Max DD:", dd_rb)

KeyError: 'Portfolio'